# 06 · R4 target-format intervention and the real LoRA trainer

R4 compared two fresh adapters: **digits** and **words**. Only the targets and matched wrong targets of 108 clock rows changed; each arm retained 2,316 rows and the original R3 exposure plan (450 optimizer updates, effective batch 32, five epochs). R3 weights were an inference comparator, never the training initialization. Ukrainian and official-corpus training rows were both zero.

## Inspect the two training runs on CPU

**Run all** reads the archived loss histories and checks target-token alignment. On a GPU, you can run two new updates on public B01 examples or repeat the full experiment with its original inputs. Held-out translations are scored separately.


In [1]:
from pathlib import Path
import json, sys, subprocess

candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p / "results/norm64_run.json").is_file()), None)
if ROOT is None:
    ROOT = Path.cwd() / "slavic-english-slm"
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "https://github.com/Silveroboros-dev/slavic-english-slm.git",
                str(ROOT),
            ],
            check=True,
        )
    if not (ROOT / "results/norm64_run.json").is_file():
        raise RuntimeError("Bootstrap checkout lacks the project results marker")
print("Repository ready")
RUN = {"mode": "archive", "output": ROOT / "r4-public-demo", "historical_inputs": None}
assert RUN["mode"] in {"archive", "public-demo", "historical-replay"}

Repository ready


## Implemented model, decoder and target formatting

The checked-in `training/train_r4.py` runs the retained training core in `r3_recipe.py`, with the original pure-Python alignment/scheduling helper. Unrelated R3 launch, validation and evaluation entry points have been removed; the actual loss/optimizer/decoder code is retained. Public code keeps checkpoints rather than deleting older ones, so budget disk space accordingly. The base is `facebook/nllb-200-distilled-600M`, pinned to revision `f8d333a098d19b4fd9a8b18f94170487ad3f821d`; seven downloaded files are size/hash verified by `model_snapshot.py`.

Source language tags are `pol_Latn`, `hrv_Latn`, `rus_Cyrl`; target mode is `eng_Latn`. Sources and targets are tokenized separately; no chat prompt, source prefix text or source-token CE is used. Targets use `text_target=...`. Truncation is forbidden; training lengths above 256 tokens fail. Targets begin with the English language token and include EOS; padding labels are `-100`.

Decoder inputs use the actual Transformers M2M100 `shift_tokens_right`: `[eng, token, EOS, -100]` becomes `[EOS, eng, token, EOS]`. Every wrong-target branch uses its own decoder prefix under the same source. Training disables cache and enables non-reentrant gradient checkpointing. The frozen base uses SDPA and CUDA autocast; trainable LoRA tensors are float32. Rank 16, alpha 32, dropout 0.05, no bias adaptation; targets are `q_proj`, `k_proj`, `v_proj`, `out_proj`, `fc1`, `fc2` in matching encoder/decoder modules.

These are source-code facts, not reviewer hypotheses. The script also tests the real decoder shift and finite forward/backward gradients at GPU startup, before optimizer updates. That smoke test is conditional and has not been rerun by this CPU notebook.


In [2]:
archive = json.loads((ROOT / "results/r4_training_archive.json").read_text())
for arm, receipt in archive["training"].items():
    history = archive["loss_history"][arm]
    assert len(history) == receipt["optimizer_updates"] == 450
    assert [x["step"] for x in history] == list(range(1, 451))
    print(
        f"{arm}: {len(history)} updates; {receipt['examples_seen']} row exposures; "
        f"first objective {history[0]['objective']:.6f}; last {history[-1]['objective']:.6f}"
    )
print("Archived GPU:", archive["runtime"]["gpu"])
print(
    "Archived libraries:",
    {k: archive["runtime"][k] for k in ["torch", "transformers", "peft", "tokenizers"]},
)

digits: 450 updates; 14400 row exposures; first objective 1.616720; last 0.739358
words: 450 updates; 14400 row exposures; first objective 1.616720; last 0.780122
Archived GPU: Tesla T4
Archived libraries: {'torch': '2.14.1+cu130', 'transformers': '4.57.1', 'peft': '0.17.1', 'tokenizers': '0.22.2'}


## Objective and optimization

The implemented objective is `sum(weight * target_CE) / valid_target_tokens + 0.1 * mean(relu(0.5 - (correct_focal_logP - wrong_focal_logP)))`. Weights are 5 on actual changed target spans and 1 elsewhere. Focal scores average conditional log probabilities over each side's changed span. `SequenceMatcher(autojunk=False)` aligns target tokens; special tokens never get focal upweighting. Pure insertion/deletion can use an adjacent unchanged token **only for a missing margin-score side**, never for CE upweighting. Ordinary conversation rows use plain CE. Both pair directions are trained.

AdamW uses LR 1e-4, weight decay 0.01, gradient clipping 1.0, linear scheduling with 5% warmup (22 of 450 updates via Python `round`). Seed is 42; TF32 is off; deterministic CUDA algorithms and `CUBLAS_WORKSPACE_CONFIG=:4096:8` are required. The R4 launcher uses microbatch 4 and accumulation 8. Overflow attempts retry without counting a completed update. Checkpoints include adapter, optimizer, scheduler, scaler and RNG states; saved receipts bind hashes and runtime.

The archived runtime reported Tesla T4 and `amp_dtype=bfloat16`. This is preserved as a recorded claim, not a corrected hardware assumption; a new run chooses BF16 only when PyTorch reports support, otherwise FP16. Numerical equivalence across stacks is not guaranteed.

Post-training inference is separate: float32, greedy (`num_beams=1`, `do_sample=False`), English forced BOS, batch 2, `max_new_tokens=256`, cache enabled and no source truncation. Loss decreases are optimization evidence, not held-out generalization or native-speaker acceptance.


In [3]:
# CPU inspection of actual code and token alignment, without torch imports.
sys.path.insert(0, str(ROOT / "training"))
import r3_recipe, r3_common

print(json.dumps(r3_recipe.TRAIN, indent=2))
left, right, anchors = r3_common.focus_alignment([10, 20, 2], [10, 30, 2], {2})
assert left == [False, True, False] and right == [False, True, False]
print("Ordered target-token replacement alignment passed")

{
  "epochs": 5,
  "effective_batch": 32,
  "lr": 0.0001,
  "weight_decay": 0.01,
  "clip_grad_norm": 1.0,
  "seed": 42,
  "lora_rank": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "target_modules": [
    "q_proj",
    "k_proj",
    "v_proj",
    "out_proj",
    "fc1",
    "fc2"
  ],
  "focus_weight": 5.0,
  "margin_nats": 0.5,
  "margin_weight": 0.1,
  "max_train_tokens": 256,
  "checkpoint_every_updates": 16,
  "epoch_policy": "Every conversation+clock row once, reviewed pair cycling at exactly 8/32 (25%), explicit conversation padding to equal language exposure. See epoch_plan.json."
}
Ordered target-token replacement alignment passed


## Optional GPU runs with public B01 examples or the original training inputs

Leave `RUN['mode']='archive'` for normal Run all. To train a **new two-update mechanics demonstration**, change it to `public-demo`. The demonstration reads public `data/b01_contrasts.jsonl`, takes 15 complete contrast pairs and adds two plain-CE controls reusing public text. It repeats these 32 rows for two optimizer updates. It uses the same actual tokenization, objective, fresh LoRA, optimizer and saving code, but a deliberately different schedule and cohort; it does not replicate R4 and generates no benchmark scores. Examples used here must not subsequently be treated as held-out evaluation. Budget approximately 2.48 GB base download plus GPU memory and optimizer/checkpoint disk space. This branch has not been executed in this CPU validation.

Use CUDA-enabled PyTorch >=2.6 and Python >=3.10, plus `transformers==4.57.1 peft==0.17.1 tokenizers==0.22.2 sentencepiece==0.2.1 safetensors`. Safetensors was not independently pinned in the original launcher; its installed version is recorded in new runtime receipts. The notebook never installs packages automatically. Review the base model's CC-BY-NC-4.0 terms before choosing a GPU mode.

For `historical-replay`, set `RUN['historical_inputs']` explicitly to the locally supplied original input directory. No private ZIP, weights or training text are distributed here. The trainer checks pinned manifest SHA-256 `6182ed7ec8cc5d4f3bcd8ecacb5d01bdb54026aa026042365ecdd305c2fc128c`, every payload file hash, the unchanged original R3 recipe/plan, matched pairs, source-only evaluation structure and no normalized holdout/source overlap. A different corpus cannot masquerade as replay. This path trains 900 updates and performs four inference runs; it is substantially larger than the demonstration.


In [4]:
if RUN["mode"] == "archive":
    print(
        "Archive mode: no installation, model download, CUDA training or private inputs requested."
    )
else:
    import os

    env = dict(os.environ, CUBLAS_WORKSPACE_CONFIG=":4096:8")
    command = [sys.executable, str(ROOT / "training/train_r4.py"), "--output", str(RUN["output"])]
    if RUN["mode"] == "public-demo":
        command += ["--public-demo", str(ROOT / "data/b01_contrasts.jsonl")]
    else:
        if RUN["historical_inputs"] is None:
            raise ValueError("Set historical_inputs explicitly for private historical replay")
        command += ["--inputs", str(RUN["historical_inputs"])]
    subprocess.run(command, env=env, check=True, cwd=ROOT)

Archive mode: no installation, model download, CUDA training or private inputs requested.


## What the training logs verify

`results/r4_training_archive.json` contains scalar loss histories and selected receipts from the actual completed results archive, with its SHA-256; no training texts or adapters. The default assertions establish history shape and executable alignment, not the truth of every historical receipt. Historical replay additionally verifies input identity and actual model files. The public demo exercises training mechanics only when explicitly selected on CUDA. Compare held-out translation outputs and independent scoring before making any quality claim.
